# TP1 — Geometría de la imagen (informe)

**Nombre:** Matías · **Entrega parte A:** _fecha_ · **Entrega parte B:** _fecha_

Este notebook **usa** `mv.geometria`. Las funciones se implementan en `mv/geometria.py`.
En los experimentos sí podés escribir código acá (por ejemplo, el *forward mapping* "malo" de A3).
Respondé en las celdas **✍️ Respuesta**.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import matplotlib.pyplot as plt
import cv2

from mv import geometria as g
from mv import imagen as im
from mv.datos import cargar_frame, mostrar

img = cargar_frame()
H, W = img.shape[:2]
print(img.shape)

---
# Parte A — Transformaciones

## A1. Ver las matrices

`dibujar_transformacion(M)` dibuja una grilla, î (rojo) y ĵ (verde) antes (gris) y después (color) de aplicar M.
**Ojo:** acá los ejes son "de matemática" (la y crece hacia arriba), no de imagen.

In [ ]:
def dibujar_transformacion(M, titulo="", ax=None):
    ax = ax or plt.gca()
    T = g.a_homogenea(np.asarray(M, float))
    for k in np.arange(-3, 4):
        for linea in (np.array([[k, -3], [k, 3]]), np.array([[-3, k], [3, k]])):
            ax.plot(*linea.T, color="0.85", lw=0.8)
            ax.plot(*g.aplicar(T, linea).T, color="tab:blue", lw=0.8, alpha=0.6)
    for v, col in (([1, 0], "tab:red"), ([0, 1], "tab:green")):
        w = g.aplicar(T, np.array([[0, 0], v]))[1]
        ax.annotate("", xy=w, xytext=(0, 0), arrowprops=dict(color=col, width=2, headwidth=8))
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal"); ax.set_title(titulo)

fig, ejes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (M, t) in zip(ejes, [(g.rotacion(np.pi / 6), "rotación 30°"), (g.escala(2, 0.5), "escala (2, 0.5)"),
                             (g.cizalla(kx=1), "cizalla kx=1"), ([[2, 1], [4, 2]], "det = 0")]):
    dibujar_transformacion(M, t, ax)
plt.show()

**Preguntas**
1. Para cada una de las 4: ¿las columnas de la matriz predicen exactamente dónde terminan î y ĵ? Mostralo con la cizalla.
2. ¿Qué le pasa a la grilla en la última? Relacionalo con el determinante y con la pregunta 5 del diagnóstico.

✍️ **Respuesta:**

## A2. Rotar un frame: lo mío contra OpenCV

In [ ]:
T = g.rotacion_alrededor(np.deg2rad(10), (W / 2, H / 2))

t0 = time.perf_counter(); mio = g.warp(img, T, (H, W)); t_mio = time.perf_counter() - t0
t0 = time.perf_counter()
ref = cv2.warpAffine(img, T[:2], (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
t_cv = time.perf_counter() - t0

dif = np.abs(mio.astype(int) - ref.astype(int))
print(f"mío: {t_mio*1000:.0f} ms | OpenCV: {t_cv*1000:.1f} ms | dif. media {dif.mean():.2f}, máx {dif.max()}")
mostrar(mio, ref, (dif.max(axis=2) > 2).astype(float), titulos=["mío", "OpenCV", "dónde difieren (>2)"], ancho=6)

**Preguntas**
3. ¿Dónde difieren las dos imágenes? ¿Por qué ahí? (Pista: mirá el docstring de `interpolar_bilineal` sobre el borde.)
4. ¿Cuántas veces más rápido es OpenCV? Dos razones posibles.
5. Con θ positivo, ¿la imagen giró en sentido horario o antihorario **en pantalla**? ¿Por qué?

✍️ **Respuesta:**

## A3. *Forward mapping*: por qué no

Ampliá un recorte ×2 **empujando** cada píxel de la entrada a su posición en la salida, redondeando.
Este código va acá en el notebook, no en `mv/`. Es el método malo, para ver por qué es malo.

In [ ]:
recorte = im.recortar(img, (W // 2 - 80, H // 2 - 60, W // 2 + 80, H // 2 + 60))
h, w = recorte.shape[:2]
T2 = g.a_homogenea(g.escala(2, 2))

# TODO: forward mapping
# 1. coordenadas (x, y) de TODOS los píxeles de `recorte` (np.indices)
# 2. aplicarles T2 con g.aplicar y redondear
# 3. crear salida = zeros (2h, 2w, 3) y asignar salida[y', x'] = recorte[y, x]
forward = np.zeros((2 * h, 2 * w, 3), dtype=np.uint8)

inverse = g.warp(recorte, T2, (2 * h, 2 * w))
mostrar(forward, inverse, titulos=["forward (empujar)", "inverse (buscar)"], ancho=7)

**Preguntas**
6. ¿Qué ves en la versión *forward*? ¿Por qué pasa? ¿Qué pasaría al **achicar** la imagen en vez de ampliarla?
7. Explicá con tus palabras por qué el *inverse mapping* no tiene ese problema.

✍️ **Respuesta:**

## A4. Vecino más cercano contra bilineal

In [ ]:
# TODO: elegí la caja de un jugador (x1, y1, x2, y2) y ampliala ×4 de las dos formas.
caja = (W // 2 - 20, H // 2 - 40, W // 2 + 20, H // 2 + 40)
jug = im.recortar(img, caja)
h, w = jug.shape[:2]
T4 = g.a_homogenea(g.escala(4, 4))
bilineal = g.warp(jug, T4, (4 * h, 4 * w))

# vecino más cercano: el mismo inverse mapping, pero redondeando las coordenadas en vez de interpolar
ys, xs = np.indices((4 * h, 4 * w))
src = g.aplicar(np.linalg.inv(T4), np.stack([xs.ravel(), ys.ravel()], 1))
vecino = jug[np.clip(np.round(src[:, 1]).astype(int), 0, h - 1),
             np.clip(np.round(src[:, 0]).astype(int), 0, w - 1)].reshape(4 * h, 4 * w, 3)

mostrar(vecino, bilineal, titulos=["vecino más cercano", "bilineal"], ancho=5)

**Preguntas**
8. ¿Qué se ve en cada uno? ¿Cuál "inventa" información? ¿Cuál conviene para una **máscara** (valores 0/1) y cuál para una foto?

✍️ **Respuesta:**

## A5. Enderezar la cancha

Marcá dos puntos sobre la línea lateral **lejana** y rotá el frame para que quede horizontal.

In [ ]:
# TODO: dos puntos (x, y) sobre la línea lateral lejana (leelos de la imagen con los ejes)
p1 = np.array([100.0, 140.0])
p2 = np.array([1100.0, 140.0])

# TODO: el ángulo de la recta (np.arctan2) y la rotación que la deja horizontal, alrededor del punto medio
angulo = ...
T_ender = ...
mostrar(img, g.warp(img, T_ender, (H, W)), titulos=["original", "enderezada"], ancho=7)

**Preguntas**
9. ¿Por qué rotar alrededor del punto medio de la línea y no del origen (0, 0)? ¿Qué pasaría con el origen?
10. Después de enderezar la lateral lejana, ¿la **cercana** también quedó horizontal? ¿Por qué no? (Esto lo resuelve la homografía en U4: una rotación no alcanza para "deshacer" la perspectiva.)

✍️ **Respuesta:**

---
# Parte B — Rectas y SVD

## B1. Ajustar la línea lateral

Marcá **~8 puntos** sobre una línea de la cancha.
- Con un frame **real**: leelos de la imagen.
- Con el frame **sintético**: hay puntos de ejemplo abajo, generados sobre las líneas que dibuja `frame_sintetico` + ruido de "click".

In [ ]:
rng = np.random.default_rng(0)
ht = H // 6                                                  # alto de la tribuna en el frame sintético

xs = np.linspace(100, W - 100, 8)
lateral_cercana = np.column_stack([xs, H - 40 - 0.08 * (xs - W / 2)]) + rng.normal(0, 1.5, (8, 2))
ys_ = np.linspace(ht + 30, H - 60, 8)
linea_medio = np.column_stack([W / 2 + 0.35 * (ys_ - ht), ys_]) + rng.normal(0, 1.5, (8, 2))

# TODO (con frame real): reemplazá por tus clicks
# lateral_cercana = np.array([[x, y], ...])
# linea_medio     = np.array([[x, y], ...])

In [ ]:
def dibujar_recta_abc(r, color, label):
    a, b, c = r
    if abs(b) > abs(a):            # más horizontal: y en función de x
        x = np.array([0, W]); y = -(a * x + c) / b
    else:                          # más vertical: x en función de y
        y = np.array([0, H]); x = -(b * y + c) / a
    plt.plot(x, y, color=color, lw=1.5, label=label)

def comparar(P, titulo):
    m, c = g.ajustar_recta_mc(P)
    r_tot = g.ajustar_recta_total(P)
    r_mc = np.array([m, -1.0, c])                                # y = m x + c  →  m x − y + c = 0
    plt.figure(figsize=(12, 7)); plt.imshow(img)
    plt.plot(*P.T, "yo", ms=6, label="clicks")
    dibujar_recta_abc(r_mc, "tab:red", "ordinarios")
    dibujar_recta_abc(r_tot, "tab:cyan", "totales")
    plt.xlim(0, W); plt.ylim(H, 0); plt.legend(); plt.title(titulo); plt.show()
    for nombre, r in (("ordinarios", r_mc), ("totales", r_tot)):
        d = g.distancia_a_recta(P, r)
        print(f"{nombre:10s}  error perpendicular RMS = {np.sqrt((d**2).mean()):.2f} px")

comparar(lateral_cercana, "B1: línea lateral")

**Preguntas**
11. ¿Las dos rectas son casi iguales? ¿Por qué, en una recta casi horizontal, minimizar el error vertical o el perpendicular da casi lo mismo?

✍️ **Respuesta:**

## B2. Recta casi vertical

In [ ]:
comparar(linea_medio, "B2: línea de medio campo")

In [ ]:
# B2 bis: el caso extremo, una línea vertical (x = 640) con 1 px de ruido de click
P_vert = np.column_stack([640 + rng.normal(0, 1, 8), np.linspace(150, 650, 8)])
print("ordinarios (m, c):", g.ajustar_recta_mc(P_vert))
print("totales (a, b, c):", g.ajustar_recta_total(P_vert).round(4))
comparar(P_vert, "B2 bis: línea vertical")

**Preguntas**
12. En la línea de medio (≈70° de inclinación) los dos métodos dan casi lo mismo, pero en la vertical los ordinarios se rompen. ¿Qué pendiente y qué ordenada dieron? ¿Por qué son esos números absurdos? Explicalo con el supuesto que hacen los ordinarios (apunte 06, sección 5).
13. Si repetís B2 bis con otra semilla del ruido, ¿la recta de los ordinarios cambia mucho o poco? ¿Y la de los totales? ¿Qué te dice eso sobre la **estabilidad** de cada método?

✍️ **Respuesta:**

## B3. Un click malo

In [ ]:
P_malo = lateral_cercana.copy()
P_malo[3, 1] -= 60                         # un click 60 px arriba: clickeaste un jugador
comparar(P_malo, "B3: un outlier")

**Preguntas**
14. ¿Cuánto se movió cada recta? ¿Alguno de los dos métodos es robusto a un *outlier*?
15. Proponé una idea (sin implementarla) para detectar automáticamente el click malo. (En U4 vamos a ver que tu idea probablemente se parezca a RANSAC.)

✍️ **Respuesta:**

## B4. Intersecciones y punto de fuga

In [ ]:
r_cercana = g.ajustar_recta_total(lateral_cercana)
r_medio = g.ajustar_recta_total(linea_medio)
esquina = g.interseccion(r_cercana, r_medio)
print("intersección lateral cercana × línea de medio:", esquina)

# TODO: ajustá también la lateral LEJANA (marcá puntos) y calculá el punto donde se cortan las dos laterales.
# lateral_lejana = ...
# fuga = g.interseccion(g.ajustar_recta_total(lateral_lejana), r_cercana)

**Preguntas**
16. ¿El punto de corte de las dos laterales está dentro o fuera de la imagen? ¿Dónde está, aproximadamente?
17. En la cancha real las laterales son **paralelas**. ¿Por qué en la imagen se cortan? Relacionalo con la división por W (apunte 04, sección 4).
18. Mové un click de la lateral lejana 2 píxeles. ¿Cuánto se mueve el punto de fuga? ¿Por qué tanto?

✍️ **Respuesta:**

## B5. La elipse de la SVD

In [ ]:
A = np.array([[2.0, 1.0], [0.5, 1.5]])
U, S, Vt = np.linalg.svd(A)
t = np.linspace(0, 2 * np.pi, 200)
circ = np.stack([np.cos(t), np.sin(t)], 1)
elipse = circ @ A.T

plt.figure(figsize=(6, 6))
plt.plot(*circ.T, "0.7", label="circunferencia"); plt.plot(*elipse.T, "tab:blue", label="A · circunferencia")
for i, col in enumerate(("tab:red", "tab:green")):
    plt.annotate("", xy=Vt[i], xytext=(0, 0), arrowprops=dict(color=col, alpha=.4, width=1.5))
    plt.annotate("", xy=S[i] * U[:, i], xytext=(0, 0), arrowprops=dict(color=col, width=2.5))
plt.gca().set_aspect("equal"); plt.legend(); plt.grid(alpha=.3)
plt.title("vᵢ (claritos) → σᵢ·uᵢ (fuertes)"); plt.show()

print("valores singulares:", S)
print("√autovalores de AᵀA:", np.sqrt(np.linalg.eigvalsh(A.T @ A))[::-1])

**Preguntas**
19. Verificá con un número que `A @ Vt[0]` es `S[0] * U[:, 0]`. ¿Qué significa geométricamente?
20. ¿Qué relación hay entre los valores singulares y |det(A)|? Verificala.

✍️ **Respuesta:**

## B6. Comprimir un frame

In [ ]:
gris = im.a_gris(img).astype(float)
s = np.linalg.svd(gris, compute_uv=False)
plt.figure(figsize=(8, 3)); plt.semilogy(s); plt.xlabel("i"); plt.ylabel("σᵢ (log)"); plt.title("valores singulares del frame"); plt.show()

ks = [1, 5, 20, 50]
aprox = [np.clip(g.aproximar_rango(gris, k), 0, 255) for k in ks]
mostrar(*aprox, titulos=[f"k = {k}" for k in ks], ancho=4)
for k in ks:
    guardado = k * (gris.shape[0] + gris.shape[1] + 1)
    print(f"k={k:3d}: {100 * guardado / gris.size:5.1f} % de los números")

**Preguntas**
21. ¿Desde qué k se reconoce el partido? ¿Y los jugadores? ¿Qué "ve" primero la aproximación y qué ve último?
22. ¿Por qué la imagen con k = 1 tiene ese aspecto de franjas horizontales y verticales? (Pista: `σ₁ u₁ v₁ᵀ` es un producto "fila × columna".)

✍️ **Respuesta:**

## Conclusiones

En 5–10 líneas: qué te llevás de U1, qué conexiones ves con el resto de la materia (homografía, Kalman, redes), y qué todavía no te cierra.

✍️ **Respuesta:**